# Initialize

In [5]:
import importlib.util
import csv
from pathlib import Path

csv_path = Path("recipe.csv")

# Resolve syringe-pump.py from common working directories
candidate_paths = [
    Path.cwd() / "syringe-pump.py",
    Path.cwd() / "devices/syringe-pump/src/syringe-pump.py",
    Path("devices/syringe-pump/src/syringe-pump.py"),
]
module_path = next((p.resolve() for p in candidate_paths if p.exists()), None)

if module_path is None:
    raise FileNotFoundError("Could not locate syringe-pump.py")

spec = importlib.util.spec_from_file_location("syringe_pump_module", module_path)
syringe_pump_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(syringe_pump_module)

# Parse CSV header to build port mappings
with csv_path.open("r", newline="", encoding="utf-8-sig") as handle:
    header = next(csv.reader(handle))

solutions = {}
for index, column_name in enumerate(header):
    if column_name.strip():
        solutions[column_name.strip().upper()] = index + 1

print("Solutions to ports mapping:")
for solution, port in solutions.items():
    print(f"  {solution}: {port}")

# Create SyringePump instance with the port mappings
pump = syringe_pump_module.SyringePump(
    solutions=solutions,
    draw_speed=2.0,
    dispense_speed=2.0,
)

print(f"\nPump initialized successfully")


Solutions to ports mapping:
  MAIN: 1
  H2O: 2
  KCL: 3
  H2O2: 4
  ETOH: 5
  ACETONE: 6
  MEOH: 7
  BUFFER: 8
  CATALYST: 9
  LIGAND: 10
  AIR: 11
  OUT: 12

Pump initialized successfully


# Add acid

In [6]:
# Execute recipe - draw and dispense acids to MAIN
recipe_result = pump.run_csv_protocol(csv_path="recipe.csv")

print("Recipe executed successfully")
for op in recipe_result["operations"]:
    print(
        f"{op['channel_label']}: {op['volume_ml']}mL from port {op['draw_valve_port']} -> MAIN"
    )

Recipe executed successfully
H2O: 10.0mL from port 2 -> MAIN
AIR: 5.0mL from port 11 -> MAIN


# Out (MAIN -> OUT)

In [7]:
out_volume_ml = 15

out_step = pump.draw_and_dispense(
    volume_ml=out_volume_ml,
    from_solution="MAIN",
    to_solution="OUT",
)

print(
    f"OUT complete | volume_mL={out_step['volume_ml']} | "
    f"draw_port={out_step['draw_valve_port']} -> dispense_port={out_step['dispense_valve_port']}"
)


OUT complete | volume_mL=15 | draw_port=1 -> dispense_port=12


# Flush (add water, then OUT)

In [ ]:
flush2out_ml = 10
water_draw_ml = 5

flush_result = pump.flush_with_solution(
    flush_solution="H2O",
    target_solution="OUT",
    flush2intermediary_volume_ml=flush2out_ml,
    flush_solution_volume_ml=water_draw_ml,
)

print(f"Flush complete")
for step in flush_result["steps"]:
    print(
        f"{step['step']} | volume_mL={step['volume_ml']} | "
        f"draw_port={step['draw_valve_port']} -> dispense_port={step['dispense_valve_port']}"
    )

Flush complete
flush_add | volume_mL=5 | draw_port=2 -> dispense_port=1
flush_out | volume_mL=10 | draw_port=1 -> dispense_port=12
